# Catch-Up Session

This session aims to cover recall related to basics of Pytorch.

Do not hesitate to ask questions!

Loading packages

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

We will define a Linear model that could be used to tackle a $\mathbb{R}^{10} \rightarrow \mathbb{R}$ supervised regression problem.

A tiny dataset is provided below.

In [ ]:
def generate_data(nb_samples=20):
  X = torch.randn(nb_samples, 10)  #
  y = X.sum(dim=-1, keepdim=True)**2 + torch.randn(nb_samples, 1) # target will simply be the square of the sum of the features + a random biais
  return X, y

X, y = generate_data()

> **Defining a model using Tensors**

First use a tensor to store the parameters - compute the prediction made considering random parameters

In [4]:
# YOUR ANSWER
w = torch.randn(10, 1, requires_grad=True)  # Initialize weights
b = torch.randn(1, requires_grad=True)      # Initialize bias
y_pred = X @ w + b      # Linear model prediction
y_pred

tensor([[-3.8284e-01],
        [-5.8099e-01],
        [-5.6349e+00],
        [ 1.3779e+00],
        [-1.6400e+00],
        [-3.1731e-01],
        [-9.7684e-01],
        [-8.1895e-01],
        [-4.8861e-03],
        [ 1.0367e+00],
        [-3.8467e+00],
        [ 7.3195e-01],
        [ 1.6869e+00],
        [-2.0697e+00],
        [-2.4466e+00],
        [-1.3320e-01],
        [ 3.9239e+00],
        [ 2.2565e+00],
        [-3.2121e+00],
        [-3.4812e+00]], grad_fn=<AddBackward0>)

> Would you be able to implement a training loop? (if yes try to do it, if not, skip this part)

In [10]:
# YOUR ANSWER
epochs = 1000
learning_rate = 0.01

for _ in range(epochs):
    loss = ((X @ w + b) - y).pow(2).mean()   # Mean Squared Error loss
    print(f"Loss: {loss.item()}")       # Print the loss value
    loss.backward()
    with torch.no_grad():
        w -= learning_rate * w.grad  # Update weights
        b -= learning_rate * b.grad  # Update bias
    w.grad.zero_()  # Reset gradients for next iteration
    b.grad.zero_()  # Reset gradients for next iteration

Loss: 72.50526428222656
Loss: 72.41246032714844
Loss: 72.3215560913086
Loss: 72.23249816894531
Loss: 72.14518737792969
Loss: 72.0595932006836
Loss: 71.97566223144531
Loss: 71.89332580566406
Loss: 71.81253051757812
Loss: 71.7332534790039
Loss: 71.65541076660156
Loss: 71.57896423339844
Loss: 71.50390625
Loss: 71.4301528930664
Loss: 71.3576889038086
Loss: 71.28646087646484
Loss: 71.21644592285156
Loss: 71.14756774902344
Loss: 71.07986450195312
Loss: 71.01324462890625
Loss: 70.94770812988281
Loss: 70.88320922851562
Loss: 70.81971740722656
Loss: 70.75721740722656
Loss: 70.69567108154297
Loss: 70.63507080078125
Loss: 70.57537078857422
Loss: 70.51656341552734
Loss: 70.4586181640625
Loss: 70.4015121459961
Loss: 70.34523010253906
Loss: 70.28974914550781
Loss: 70.23503112792969
Loss: 70.18109130859375
Loss: 70.12788391113281
Loss: 70.0754165649414
Loss: 70.02365112304688
Loss: 69.97257995605469
Loss: 69.92219543457031
Loss: 69.87247467041016
Loss: 69.82337951660156
Loss: 69.77493286132812
Loss: 

> **Modelling choices: Loss**

Select a loss that is relevant for the problem under study and compute it for the predictions made by the model.

In [15]:
loss_fct = nn.MSELoss()
loss_fct(y, y_pred)  # Mean Squared Error loss

tensor(240.4590, grad_fn=<MseLossBackward0>)

> What would be your choice of loss if you had to deal with a classification task?

**Answer**: Cross-Entropy Loss

> **Defining a model using nn.Sequential**

Now use ```nn.Sequential``` to define the same linear model

In [16]:
model = nn.Sequential(
    nn.Linear(10, 1)
)
y_pred = model(X)
loss_fct(y, y_pred)  # Mean Squared Error loss

tensor(240.0274, grad_fn=<MseLossBackward0>)

> Make sure the model uses a biais



> **Defining a model extending nn.Module**
Define the exact same model as a Python class extending ```nn.Module```

In [ ]:
class MyLinearModel(nn.Module):
    def __init__(self, input_size=10, output_size=1):
        super(MyLinearModel, self).__init__()
        self.linear = nn.Linear(input_size, output_size)

    def forward(self, x):
        output = self.linear(x)
        return output

# Initialize the model
model = MyLinearModel(input_size=10, output_size=1).to("mps" if torch.backends.mps.is_available() else "cpu")

In [ ]:
# YOUR ANSWER

Note how device are managed in Pytorch, e.g. to use GPU to compute faster.

In [ ]:
device = "mps" if torch.backend.mps.is_available() else "cpu"
model = model.to(device)
for xb, yb in loader:
    xb, yb = xb.to(device), yb.to(device)

* Model and data must be on the same device.
* Load checkpoints with torch.load(path, map_location=device), or a GPU-saved file will fail on a CPU-only machine (model saving is discussed later in this session).

> **MLP Implementation**

Define the architecture of a Multi-Layer Perceptron (MLP) to solve the same task (single hidden layer with 64 neurons using ReLU).

In [27]:
class MyMLP(nn.Module):
    def __init__(self, input_size=10, hidden_size=64, output_size=1):
        super(MyMLP, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.activation = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        output = self.fc1(x)
        output = self.activation(output)
        output = self.fc2(output)

        return output


my_mlp = MyMLP(input_size=10, hidden_size=64, output_size=1).to("mps" if torch.backends.mps.is_available() else "cpu")

> **Training Step**

Implement one training step of the MLP using ```optim.SGD``` optimizer.

First instantiate the optimizer specifying the model parameters to be optimized.

Then recall the substeps of a step:
1. Forward pass (prediction)
2. Loss computation
3. Set the optimizer gradient to 0
4. Compute the gradient (backpropagation)
5. Tell the optimizer to use the gradient to update the parameter values (step)








In [33]:
learning_rate = 1e-2
device = "mps"
optimizer = optim.SGD(params=my_mlp.parameters(), lr=learning_rate)
criterion = nn.MSELoss()
epochs = 100

In [29]:
def train_one_epoch(model, criterion, optimizer, training_data, device) -> float:
    model.train()
    optimizer.zero_grad()

    X, y = training_data
    X, y = X.to_device(device), y.to_device(device)

    y_pred = model(X)
    loss = criterion(y_pred, y)
    loss.backward()
    optimizer.step()

    return loss

> **Training Loop**

Create a loop to make several steps with a run

In [30]:
def fit(model, epochs, criterion, optimizer, training_data, device, log_freq) -> None:
    for i in range(1, epochs + 1):
        epoch_loss = train_one_epoch(model, criterion, optimizer, training_data, device)

        if (i % log_freq) == 0:
            print(f"Epoch {i}: Loss = {epoch_loss:.4f}")

> What would have to be modified to deal with a claffication task.

**Answer**
1. The criterion (loss function) should be nn.CrossEntropyLoss
2. The last layer could have to be adapted to support the number of classes in our problem (if 
we just have one, we would not be obligated to adapt)

> **Dataset**

Consider now that we are dealing with a larger dataset

In [31]:
X, y = generate_data(200_000)

You'd not be able to execute the forward pass on such a big chunk of data, e.g. ```model(X)``` would crash.

> Use ```TensorDataset``` and ```DataLoader``` to iterate using batches. Also create a validation set and compute the loss on this set (using ```random_split```).

In [ ]:
from torch.utils.data import DataLoader, TensorDataset, random_split

dataset = TensorDataset(X, y)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)

> Modify the training code

In [ ]:
# YOUR ANSWER

> This corresponds to a single epoch.
Now add an extra loop to allow several epochs and use the validation data to monitor the validation loss

In [ ]:
# YOUR ANSWER


> **Tensorboard**

*TensorBoard is a visualization toolkit for machine learning experimentation. TensorBoard allows tracking and visualizing metrics such as loss and accuracy, visualizing the model graph, viewing histograms, displaying images and much more.*
https://docs.pytorch.org/tutorials/recipes/recipes/tensorboard_with_pytorch.html

Use tensorboard to monitor the loss using ```SummaryWriter```

In [ ]:
# YOUR ANSWER


In [ ]:
# Load the TensorBoard notebook extension
%load_ext tensorboard

In [ ]:
%tensorboard --logdir=runs

# **Custom Datasets using data.Dataset**

> You can also extend Dataset to create custom datasets

In [ ]:
class NoisyLine(torch.utils.data.Dataset):
    def __init__(self, n=1000):
        self.x = torch.rand(n, 1)
        self.y = 3 * self.x + 1 + 0.1 * torch.randn(n, 1)
    def __len__(self):
        return len(self.x)
    def __getitem__(self, i):
        return self.x[i], self.y[i]

> What is the benefit of this approach?




**Answer**:

> **Save / Load models**

Have a look on how to save and load a model
https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html

Make sure you are able to save a model and load it afterward.
What if you'd like to resume training.  

In [ ]:
# YOUR ANSWER
# Save and Load for Inference Only


In [ ]:
# YOUR ANSWER
# Save and Load to Resume Training


> **Model Composition**

Define a new model that is based on two instances of your MLP class
e.g. $\hat{y} = MLP_2(MLP_1(x))$

In [ ]:
# YOUR ANSWER


> **Skip connection**

A skip connection (resildual or shortcut connection)  is used to bypass one or more layers by feeding the output of one layer directly as an input to a later layer.

Consider this simple example, with $j=i+1$, $x_i \in \mathbb{R}^d$, $z_j \in \mathbb{R}^d$ the output of layer $j$ with $F_j : \mathbb{R}^d â†’ \mathbb{R}^d$ one of its component (e.g. classical hidden layer) and $z_j= F_j(x_i) + x_i$.

Try implementing such a skip connection modifying one of your MLP.

In [ ]:
# YOUR ANSWER

> **CNN**

Build a CNN for (N,1,28,28) inputs.

Give the output shape after each block and the total parameter count.

In [ ]:
# YOUR ANSWER

**Answer**

| Layer |	Parameters |
|---|---|

**StepLR scheduler and Gradient Clipping**

Add AdamW, a StepLR scheduler and gradient clipping to your training loop. Where does each call go?

*The PyTorch StepLR Documentation defines torch.optim.lr_scheduler.StepLR as a learning rate scheduler that decays the learning rate of each parameter group by a multiplicative factor gamma every step_size epochs.*
https://docs.pytorch.org/docs/2.14/generated/torch.optim.lr_scheduler.StepLR.html

torch.nn.utils.clip_grad_norm_ scales down the gradients of model parameters in-place so that their total norm does not exceed a specified maximum value.

https://docs.pytorch.org/docs/2.14/generated/torch.nn.utils.clip_grad_norm_.html

In [ ]:
# YOUR ANSWER

**Freezing Layers**

Replace the head of a pretrained ResNet18 and train only the head.
Use torchvision to load the model.

In [ ]:
# YOUR ANSWER